# Credit Scoring Enhanced — Tích hợp Bảng Phụ

## Mục tiêu
- Xây dựng mô hình credit scoring **nâng cao** bằng cách tích hợp features từ **4 bảng dữ liệu**
- So sánh: Notebook gốc (`credit_scoring_no_history.ipynb`) chỉ dùng `application_train.csv` (48 features, AUC 0.767)
- Notebook này bổ sung thêm 66 features từ 3 bảng phụ → **114 features, AUC ~0.781**

## Nguồn dữ liệu

| # | Bảng | Rows | Mô tả | Features tạo |
|---|------|------|--------|-------------|
| 1 | `application_train.csv` | 307K | Hồ sơ đăng ký vay (**chính**) | 48 (34 raw + 19 engineered - 5 DAYS) |
| 2 | `previous_application.csv` | 1.67M | Lịch sử nộp đơn vay trước đó | 22 features |
| 3 | `installments_payments.csv` | 13.6M | Lịch sử trả từng kỳ | 24 features |
| 4 | `POS_CASH_balance.csv` | 10M | Số dư POS/tiền mặt theo tháng | 20 features |

## Pipeline
1. Load & Select features từ application_train
2. Load 3 bảng phụ → aggregate features cấp khách hàng
3. Data Cleaning
4. Feature Engineering (financial ratios + EXT_SOURCE combos)
5. Encode Categorical
6. Train LightGBM (5-Fold CV)
7. SHAP Explainability
8. FICO Scoring & Calibration
9. Save Artifacts

## 1. Import Libraries & Load Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json
import joblib
import os
import gc
from pathlib import Path

from lightgbm import LGBMClassifier
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
plt.style.use('seaborn-v0_8-whitegrid')

print('Libraries loaded successfully!')

In [ ]:
# Load application_train.csv
os.chdir(os.path.dirname(os.path.abspath('__file__')))
DATA_PATH = '../input/application_train.csv' if os.path.exists('../input/application_train.csv') else 'input/application_train.csv'
INPUT_DIR = os.path.dirname(DATA_PATH)

df = pd.read_csv(DATA_PATH)
print(f'application_train shape: {df.shape}')
print(f'Target distribution:\n{df["TARGET"].value_counts(normalize=True).round(4)}')
df.head()

## 2. Feature Selection — 34 Raw Features từ Application

In [ ]:
# ============================================================
# FEATURE SELECTION: 7 nhóm features
# ============================================================

# Nhóm 1: Nhân khẩu học
DEMOGRAPHIC_FEATURES = [
    'CODE_GENDER', 'DAYS_BIRTH', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS',
    'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
]

# Nhóm 2: Quy mô khoản vay & Dòng tiền
FINANCIAL_FEATURES = [
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY',
    'AMT_GOODS_PRICE', 'NAME_CONTRACT_TYPE',
]

# Nhóm 3: Việc làm & Thâm niên
EMPLOYMENT_FEATURES = [
    'DAYS_EMPLOYED', 'NAME_INCOME_TYPE', 'OCCUPATION_TYPE',
    'ORGANIZATION_TYPE', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH',
]

# Nhóm 4: Tài sản (Proxy)
ASSET_FEATURES = ['FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'OWN_CAR_AGE']

# Nhóm 5: Khu vực & Nhà ở
REGION_FEATURES = [
    'NAME_HOUSING_TYPE', 'REGION_RATING_CLIENT',
    'REGION_RATING_CLIENT_W_CITY', 'REGION_POPULATION_RELATIVE',
]

# Nhóm 6: Điểm tín dụng thay thế
EXTERNAL_FEATURES = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

# Nhóm 7: Mạng lưới xã hội & Liên lạc
SOCIAL_FEATURES = [
    'DEF_30_CNT_SOCIAL_CIRCLE', 'DEF_60_CNT_SOCIAL_CIRCLE',
    'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL',
    'DAYS_LAST_PHONE_CHANGE',
]

RAW_FEATURES = (DEMOGRAPHIC_FEATURES + FINANCIAL_FEATURES +
                EMPLOYMENT_FEATURES + ASSET_FEATURES +
                REGION_FEATURES + EXTERNAL_FEATURES + SOCIAL_FEATURES)

print(f'Tổng số raw features: {len(RAW_FEATURES)}')
for name, feats in [('Nhân khẩu học', DEMOGRAPHIC_FEATURES),
                     ('Quy mô & Dòng tiền', FINANCIAL_FEATURES),
                     ('Việc làm & Thâm niên', EMPLOYMENT_FEATURES),
                     ('Tài sản (Proxy)', ASSET_FEATURES),
                     ('Khu vực & Nhà ở', REGION_FEATURES),
                     ('Điểm thay thế', EXTERNAL_FEATURES),
                     ('Mạng lưới XH & Liên lạc', SOCIAL_FEATURES)]:
    print(f'  {name}: {len(feats)} features')

## 3. Bổ sung Features từ Bảng Phụ

Aggregate features cấp `SK_ID_CURR` từ 3 bảng phụ → merge vào `df`.

| Bảng | Features | Coverage |
|------|----------|----------|
| `previous_application` | 22 | ~94.6% |
| `installments_payments` | 24 | ~94.8% |
| `POS_CASH_balance` | 20 | ~94.1% |

In [ ]:
# ============================================================
# LOAD & ENGINEER FEATURES: previous_application
# ============================================================
prev = pd.read_csv(f"{INPUT_DIR}/previous_application.csv")
print(f"previous_application: {prev.shape}")
print(f"  Unique SK_ID_CURR: {prev['SK_ID_CURR'].nunique():,}")
print(f"  Coverage in train: {df['SK_ID_CURR'].isin(prev['SK_ID_CURR']).mean():.1%}")

# Replace sentinel values
for col in ['DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION',
            'DAYS_LAST_DUE', 'DAYS_TERMINATION']:
    prev[col].replace(365243, np.nan, inplace=True)

# Engineered ratios
prev['APP_CREDIT_PERC'] = prev['AMT_APPLICATION'] / prev['AMT_CREDIT'].replace(0, np.nan)
prev['CREDIT_TO_ANNUITY'] = prev['AMT_CREDIT'] / prev['AMT_ANNUITY'].replace(0, np.nan)

# Aggregations
prev_agg = prev.groupby('SK_ID_CURR').agg(
    PREV_APP_COUNT=('SK_ID_PREV', 'nunique'),
    PREV_AMT_CREDIT_MEAN=('AMT_CREDIT', 'mean'),
    PREV_AMT_CREDIT_MAX=('AMT_CREDIT', 'max'),
    PREV_AMT_CREDIT_SUM=('AMT_CREDIT', 'sum'),
    PREV_AMT_ANNUITY_MEAN=('AMT_ANNUITY', 'mean'),
    PREV_AMT_ANNUITY_MAX=('AMT_ANNUITY', 'max'),
    PREV_AMT_APPLICATION_MEAN=('AMT_APPLICATION', 'mean'),
    PREV_AMT_DOWN_PAYMENT_MEAN=('AMT_DOWN_PAYMENT', 'mean'),
    PREV_RATE_DOWN_PAYMENT_MEAN=('RATE_DOWN_PAYMENT', 'mean'),
    PREV_CNT_PAYMENT_MEAN=('CNT_PAYMENT', 'mean'),
    PREV_DAYS_DECISION_MEAN=('DAYS_DECISION', 'mean'),
    PREV_DAYS_DECISION_MAX=('DAYS_DECISION', 'max'),
    PREV_APP_CREDIT_PERC_MEAN=('APP_CREDIT_PERC', 'mean'),
    PREV_APP_CREDIT_PERC_VAR=('APP_CREDIT_PERC', 'var'),
    PREV_CREDIT_TO_ANNUITY_MEAN=('CREDIT_TO_ANNUITY', 'mean'),
).reset_index()

# Approval/Refused sub-segments
for status, prefix in [('Approved', 'PREV_APPROVED'), ('Refused', 'PREV_REFUSED')]:
    sub = prev[prev['NAME_CONTRACT_STATUS'] == status]
    sub_agg = sub.groupby('SK_ID_CURR').agg(
        COUNT=('SK_ID_PREV', 'nunique'),
        AMT_CREDIT_MEAN=('AMT_CREDIT', 'mean'),
        APP_CREDIT_PERC_MEAN=('APP_CREDIT_PERC', 'mean'),
    ).reset_index()
    sub_agg.columns = ['SK_ID_CURR'] + [f'{prefix}_{c}' for c in sub_agg.columns[1:]]
    prev_agg = prev_agg.merge(sub_agg, on='SK_ID_CURR', how='left')

prev_agg['PREV_APPROVAL_RATE'] = (
    prev_agg['PREV_APPROVED_COUNT'].fillna(0) / prev_agg['PREV_APP_COUNT']
)

df = df.merge(prev_agg, on='SK_ID_CURR', how='left')
prev_features = [c for c in prev_agg.columns if c != 'SK_ID_CURR']
print(f"\n✅ {len(prev_features)} features từ previous_application")
print(f"   NaN rate: {df[prev_features[0]].isna().mean():.1%}")
del prev, prev_agg
gc.collect()
print(f"   Dataset shape: {df.shape}")

In [ ]:
# ============================================================
# LOAD & ENGINEER FEATURES: installments_payments
# ============================================================
ins = pd.read_csv(f"{INPUT_DIR}/installments_payments.csv")
print(f"installments_payments: {ins.shape}")
print(f"  Unique SK_ID_CURR: {ins['SK_ID_CURR'].nunique():,}")
print(f"  Coverage in train: {df['SK_ID_CURR'].isin(ins['SK_ID_CURR']).mean():.1%}")

# Engineered per-installment features
ins['PAYMENT_PERC'] = ins['AMT_PAYMENT'] / ins['AMT_INSTALMENT'].replace(0, np.nan)
ins['PAYMENT_DIFF'] = ins['AMT_INSTALMENT'] - ins['AMT_PAYMENT']
ins['DPD'] = (ins['DAYS_ENTRY_PAYMENT'] - ins['DAYS_INSTALMENT']).clip(lower=0)
ins['DBD'] = (ins['DAYS_INSTALMENT'] - ins['DAYS_ENTRY_PAYMENT']).clip(lower=0)
ins['LATE_PAYMENT'] = (ins['DPD'] > 0).astype(int)
ins['DPD_7'] = (ins['DPD'] >= 7).astype(int)
ins['DPD_15'] = (ins['DPD'] >= 15).astype(int)

# Aggregations
ins_agg = ins.groupby('SK_ID_CURR').agg(
    INSTAL_COUNT=('SK_ID_PREV', 'count'),
    INSTAL_NUM_LOANS=('SK_ID_PREV', 'nunique'),
    INSTAL_DPD_MAX=('DPD', 'max'),
    INSTAL_DPD_MEAN=('DPD', 'mean'),
    INSTAL_DPD_SUM=('DPD', 'sum'),
    INSTAL_DBD_MAX=('DBD', 'max'),
    INSTAL_DBD_MEAN=('DBD', 'mean'),
    INSTAL_PAYMENT_PERC_MEAN=('PAYMENT_PERC', 'mean'),
    INSTAL_PAYMENT_PERC_VAR=('PAYMENT_PERC', 'var'),
    INSTAL_PAYMENT_DIFF_MEAN=('PAYMENT_DIFF', 'mean'),
    INSTAL_PAYMENT_DIFF_SUM=('PAYMENT_DIFF', 'sum'),
    INSTAL_AMT_INSTALMENT_MAX=('AMT_INSTALMENT', 'max'),
    INSTAL_AMT_INSTALMENT_MEAN=('AMT_INSTALMENT', 'mean'),
    INSTAL_AMT_PAYMENT_MEAN=('AMT_PAYMENT', 'mean'),
    INSTAL_AMT_PAYMENT_SUM=('AMT_PAYMENT', 'sum'),
    INSTAL_LATE_PAYMENT_RATE=('LATE_PAYMENT', 'mean'),
    INSTAL_LATE_PAYMENT_COUNT=('LATE_PAYMENT', 'sum'),
    INSTAL_DPD_7_RATE=('DPD_7', 'mean'),
    INSTAL_DPD_15_RATE=('DPD_15', 'mean'),
).reset_index()

# Recent behavior: last 365 days
ins_recent = ins[ins['DAYS_INSTALMENT'] >= -365]
if len(ins_recent) > 0:
    ins_recent_agg = ins_recent.groupby('SK_ID_CURR').agg(
        INSTAL_365_DPD_MAX=('DPD', 'max'),
        INSTAL_365_DPD_MEAN=('DPD', 'mean'),
        INSTAL_365_LATE_RATE=('LATE_PAYMENT', 'mean'),
        INSTAL_365_PAYMENT_PERC_MEAN=('PAYMENT_PERC', 'mean'),
        INSTAL_365_PAYMENT_DIFF_MEAN=('PAYMENT_DIFF', 'mean'),
    ).reset_index()
    ins_agg = ins_agg.merge(ins_recent_agg, on='SK_ID_CURR', how='left')

df = df.merge(ins_agg, on='SK_ID_CURR', how='left')
ins_features = [c for c in ins_agg.columns if c != 'SK_ID_CURR']
print(f"\n✅ {len(ins_features)} features từ installments_payments")
print(f"   NaN rate: {df[ins_features[0]].isna().mean():.1%}")
del ins, ins_agg, ins_recent
gc.collect()
print(f"   Dataset shape: {df.shape}")

In [ ]:
# ============================================================
# LOAD & ENGINEER FEATURES: POS_CASH_balance
# ============================================================
pos = pd.read_csv(f"{INPUT_DIR}/POS_CASH_balance.csv")
print(f"POS_CASH_balance: {pos.shape}")
print(f"  Unique SK_ID_CURR: {pos['SK_ID_CURR'].nunique():,}")
print(f"  Coverage in train: {df['SK_ID_CURR'].isin(pos['SK_ID_CURR']).mean():.1%}")

# Engineered per-record features
pos['POS_IS_DPD'] = (pos['SK_DPD'] > 0).astype(int)
pos['POS_IS_DPD_UNDER_120'] = ((pos['SK_DPD'] > 0) & (pos['SK_DPD'] < 120)).astype(int)
pos['POS_IS_DPD_OVER_120'] = (pos['SK_DPD'] >= 120).astype(int)
pos['POS_IS_COMPLETED'] = (pos['NAME_CONTRACT_STATUS'] == 'Completed').astype(int)
pos['POS_IS_ACTIVE'] = (pos['NAME_CONTRACT_STATUS'] == 'Active').astype(int)

# Aggregations
pos_agg = pos.groupby('SK_ID_CURR').agg(
    POS_RECORD_COUNT=('SK_ID_PREV', 'count'),
    POS_NUM_LOANS=('SK_ID_PREV', 'nunique'),
    POS_MONTHS_BALANCE_MAX=('MONTHS_BALANCE', 'max'),
    POS_MONTHS_BALANCE_MEAN=('MONTHS_BALANCE', 'mean'),
    POS_DPD_MAX=('SK_DPD', 'max'),
    POS_DPD_MEAN=('SK_DPD', 'mean'),
    POS_DPD_SUM=('SK_DPD', 'sum'),
    POS_DPD_DEF_MAX=('SK_DPD_DEF', 'max'),
    POS_DPD_DEF_MEAN=('SK_DPD_DEF', 'mean'),
    POS_CNT_INSTALMENT_MEAN=('CNT_INSTALMENT', 'mean'),
    POS_CNT_INSTALMENT_FUTURE_MEAN=('CNT_INSTALMENT_FUTURE', 'mean'),
    POS_IS_DPD_RATE=('POS_IS_DPD', 'mean'),
    POS_IS_DPD_SUM=('POS_IS_DPD', 'sum'),
    POS_DPD_UNDER_120_RATE=('POS_IS_DPD_UNDER_120', 'mean'),
    POS_DPD_OVER_120_RATE=('POS_IS_DPD_OVER_120', 'mean'),
    POS_COMPLETED_RATE=('POS_IS_COMPLETED', 'mean'),
    POS_ACTIVE_RATE=('POS_IS_ACTIVE', 'mean'),
).reset_index()

# Loan-level metrics
pos_loan = pos.groupby(['SK_ID_CURR', 'SK_ID_PREV']).agg(
    loan_completed=('POS_IS_COMPLETED', 'max'),
    loan_dpd_max=('SK_DPD', 'max'),
    loan_months=('MONTHS_BALANCE', 'count'),
).reset_index()

pos_loan_agg = pos_loan.groupby('SK_ID_CURR').agg(
    POS_LOAN_COMPLETED_MEAN=('loan_completed', 'mean'),
    POS_LOAN_DPD_MAX_MEAN=('loan_dpd_max', 'mean'),
    POS_LOAN_DURATION_MEAN=('loan_months', 'mean'),
).reset_index()

pos_agg = pos_agg.merge(pos_loan_agg, on='SK_ID_CURR', how='left')

df = df.merge(pos_agg, on='SK_ID_CURR', how='left')
pos_features = [c for c in pos_agg.columns if c != 'SK_ID_CURR']
print(f"\n✅ {len(pos_features)} features từ POS_CASH_balance")
print(f"   NaN rate: {df[pos_features[0]].isna().mean():.1%}")
del pos, pos_agg, pos_loan, pos_loan_agg
gc.collect()

# Summary
all_new_features = prev_features + ins_features + pos_features
print(f"\n{'='*60}")
print(f"TỔNG KẾT: {len(all_new_features)} features mới từ 3 bảng phụ")
print(f"  previous_application: {len(prev_features)}")
print(f"  installments_payments: {len(ins_features)}")
print(f"  POS_CASH_balance: {len(pos_features)}")
print(f"  Dataset shape: {df.shape}")
print(f"{'='*60}")

## 4. Data Cleaning

In [ ]:
# ============================================================
# FILTER & CLEAN — giống notebook gốc + merge new features
# ============================================================

# Chỉ giữ RAW_FEATURES + new features + SK_ID_CURR + TARGET
needed_cols = list(set(['SK_ID_CURR', 'TARGET'] + RAW_FEATURES + all_new_features))
needed_cols = [c for c in needed_cols if c in df.columns]
data = df[needed_cols].copy()
print(f'Selected columns: {data.shape}')

# Cleaning giống hệt notebook gốc
data = data[data['CODE_GENDER'] != 'XNA']
data['DAYS_EMPLOYED'] = data['DAYS_EMPLOYED'].replace(365243, np.nan)
data['DAYS_LAST_PHONE_CHANGE'] = data['DAYS_LAST_PHONE_CHANGE'].replace(0, np.nan)
data = data[data['AMT_INCOME_TOTAL'] < 20_000_000]

print(f'After cleaning: {data.shape}')
missing = data.isnull().sum()
missing_pct = (missing / len(data) * 100).round(1)
missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct})
print(f'\nMissing values (top 10):')
print(missing_df[missing_df['count'] > 0].sort_values('pct', ascending=False).head(10))

## 5. Feature Engineering — Financial Ratios & EXT_SOURCE Combos

In [ ]:
# ============================================================
# FEATURE ENGINEERING — giống hệt notebook gốc
# ============================================================

# Nhóm 1: Chuyển đổi đơn vị thời gian
data['AGE_YEARS'] = (-data['DAYS_BIRTH'] / 365.25).round(1)
data['EMPLOYMENT_YEARS'] = (-data['DAYS_EMPLOYED'] / 365.25).round(1)
data['REGISTRATION_YEARS'] = (-data['DAYS_REGISTRATION'] / 365.25).round(1)
data['ID_PUBLISH_YEARS'] = (-data['DAYS_ID_PUBLISH'] / 365.25).round(1)
data['PHONE_CHANGE_DAYS'] = -data['DAYS_LAST_PHONE_CHANGE']

# Nhóm 2: Tỷ số tài chính
data['CREDIT_INCOME_RATIO'] = data['AMT_CREDIT'] / data['AMT_INCOME_TOTAL']
data['ANNUITY_INCOME_RATIO'] = data['AMT_ANNUITY'] / data['AMT_INCOME_TOTAL']
data['CREDIT_TERM_MONTHS'] = data['AMT_CREDIT'] / data['AMT_ANNUITY']
data['PAYMENT_RATE'] = data['AMT_ANNUITY'] / data['AMT_CREDIT']
data['INCOME_PER_PERSON'] = data['AMT_INCOME_TOTAL'] / data['CNT_FAM_MEMBERS']
data['GOODS_CREDIT_RATIO'] = data['AMT_GOODS_PRICE'] / data['AMT_CREDIT']

# Nhóm 3: Tỷ số ổn định
data['EMPLOYED_TO_AGE_RATIO'] = data['DAYS_EMPLOYED'] / data['DAYS_BIRTH']

# Nhóm 4: External Source
data['EXT_SOURCE_MEAN'] = data[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1)
data['EXT_SOURCE_PROD'] = data['EXT_SOURCE_1'] * data['EXT_SOURCE_2'] * data['EXT_SOURCE_3']
data['EXT_SOURCE_MIN'] = data[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].min(axis=1)
data['EXT_SOURCE_MAX'] = data[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].max(axis=1)

# Nhóm 5: Social risk
data['SOCIAL_DEF_TOTAL'] = data['DEF_30_CNT_SOCIAL_CIRCLE'] + data['DEF_60_CNT_SOCIAL_CIRCLE']

# Nhóm 6: Phân nhóm tuổi
def age_group(age):
    if age < 27: return 0
    elif age < 35: return 1
    elif age < 45: return 2
    elif age < 55: return 3
    elif age < 65: return 4
    else: return 5
data['AGE_GROUP'] = data['AGE_YEARS'].apply(age_group)

# Nhóm 7: Contact richness
contact_cols = ['FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL']
data['CONTACT_COUNT'] = data[contact_cols].sum(axis=1)

engineered = ['AGE_YEARS', 'EMPLOYMENT_YEARS', 'REGISTRATION_YEARS', 'ID_PUBLISH_YEARS',
              'PHONE_CHANGE_DAYS', 'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO',
              'CREDIT_TERM_MONTHS', 'PAYMENT_RATE', 'INCOME_PER_PERSON',
              'GOODS_CREDIT_RATIO', 'EMPLOYED_TO_AGE_RATIO', 'EXT_SOURCE_MEAN',
              'EXT_SOURCE_PROD', 'EXT_SOURCE_MIN', 'EXT_SOURCE_MAX',
              'SOCIAL_DEF_TOTAL', 'AGE_GROUP', 'CONTACT_COUNT']

print(f'After feature engineering: {data.shape}')
print(f'{len(engineered)} engineered features + {len(all_new_features)} from supplementary tables')

## 6. Encode Categorical Features

In [ ]:
# ============================================================
# ENCODE CATEGORICAL — LabelEncoder (LightGBM native support)
# ============================================================
cat_cols = data.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns ({len(cat_cols)}): {cat_cols}')

label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    data[col] = data[col].fillna('MISSING')
    data[col] = le.fit_transform(data[col])
    label_encoders[col] = le
    print(f'  {col}: {len(le.classes_)} categories')

print(f'\nAll encoded. Dataset shape: {data.shape}')

## 7. Final Feature Set & EDA

In [ ]:
# ============================================================
# FINAL FEATURES: 48 baseline + 66 supplementary = 114
# ============================================================
DROP_COLS = ['SK_ID_CURR', 'TARGET',
             'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION',
             'DAYS_ID_PUBLISH', 'DAYS_LAST_PHONE_CHANGE']

FINAL_FEATURES = [col for col in data.columns if col not in DROP_COLS]

# Phân loại features theo nguồn
baseline_feats = [f for f in FINAL_FEATURES if f not in all_new_features]
print(f'Final feature count: {len(FINAL_FEATURES)}')
print(f'  Baseline (application): {len(baseline_feats)}')
print(f'  previous_application:   {len([f for f in FINAL_FEATURES if f in prev_features])}')
print(f'  installments_payments:  {len([f for f in FINAL_FEATURES if f in ins_features])}')
print(f'  POS_CASH_balance:       {len([f for f in FINAL_FEATURES if f in pos_features])}')

In [ ]:
# ---- EDA: Correlation với TARGET ----
correlations = data[FINAL_FEATURES + ['TARGET']].corr()['TARGET'].drop('TARGET').sort_values()

# Top 20 (absolute)
top_corr = correlations.abs().sort_values(ascending=False).head(20)
print('Top 20 features tương quan mạnh nhất (|correlation|):')
for i, (feat, corr) in enumerate(top_corr.items(), 1):
    source = ('PREV' if feat in prev_features else
              'INSTAL' if feat in ins_features else
              'POS' if feat in pos_features else 'APP')
    direction = '↓ risk' if correlations[feat] < 0 else '↑ risk'
    print(f'  {i:2d}. [{source:6s}] {feat}: {correlations[feat]:+.4f} ({direction})')

## 8. Train LightGBM — 5-Fold Stratified CV

In [ ]:
# ============================================================
# PREPARE DATA
# ============================================================
X = data[FINAL_FEATURES].copy()
y = data['TARGET'].copy()

print(f'X shape: {X.shape}')
print(f'y distribution: {y.value_counts().to_dict()}')
print(f'Positive rate: {y.mean():.4f}')

In [ ]:
# ============================================================
# 5-FOLD STRATIFIED CROSS VALIDATION
# ============================================================

N_FOLDS = 5
folds = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
feature_importance_df = pd.DataFrame()
fold_aucs = []

lgbm_params = dict(
    n_estimators=1000,
    max_depth=6,
    num_leaves=31,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=1.0,
    min_child_samples=30,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
    is_unbalance=True,
)

best_model = None
best_auc = 0

for fold_n, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = LGBMClassifier(**lgbm_params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        eval_metric='auc',
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, first_metric_only=True, verbose=False),
            lgb.log_evaluation(period=0),
        ]
    )

    val_preds = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_preds

    fold_auc = roc_auc_score(y_val, val_preds)
    fold_aucs.append(fold_auc)
    print(f'Fold {fold_n + 1}: AUC = {fold_auc:.6f} | Best iter: {model.best_iteration_}')

    fold_imp = pd.DataFrame({
        'feature': FINAL_FEATURES,
        'importance': model.feature_importances_,
        'fold': fold_n + 1
    })
    feature_importance_df = pd.concat([feature_importance_df, fold_imp])

    if fold_auc > best_auc:
        best_auc = fold_auc
        best_model = model

overall_auc = roc_auc_score(y, oof_preds)
print(f'\n{"="*50}')
print(f'Overall OOF AUC: {overall_auc:.6f}')
print(f'Mean Fold AUC:   {np.mean(fold_aucs):.6f} ± {np.std(fold_aucs):.6f}')
print(f'Best fold model saved (AUC: {best_auc:.6f})')
print(f'{"="*50}')

## 9. Feature Importance Analysis — By Source Table

In [ ]:
# ---- Feature Importance: Top 30 color-coded by source ----
imp_mean = (feature_importance_df
            .groupby('feature')['importance']
            .mean()
            .sort_values(ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(18, 10))

# Left: Top 30 overall
top30 = imp_mean.head(30)
colors_top = []
for f in top30.index:
    if f in prev_features:
        colors_top.append('#e74c3c')   # Red
    elif f in ins_features:
        colors_top.append('#3498db')   # Blue
    elif f in pos_features:
        colors_top.append('#2ecc71')   # Green
    else:
        colors_top.append('#95a5a6')   # Gray = baseline

axes[0].barh(range(len(top30)), top30.values[::-1], color=colors_top[::-1])
axes[0].set_yticks(range(len(top30)))
axes[0].set_yticklabels(top30.index[::-1], fontsize=9)
axes[0].set_xlabel('Mean Feature Importance')
axes[0].set_title('Top 30 Features (Enhanced Model)')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#95a5a6', label='Baseline (application)'),
    Patch(facecolor='#e74c3c', label='NEW: previous_application'),
    Patch(facecolor='#3498db', label='NEW: installments_payments'),
    Patch(facecolor='#2ecc71', label='NEW: POS_CASH_balance'),
]
axes[0].legend(handles=legend_elements, loc='lower right', fontsize=8)

# Right: Importance by source group
group_imp = {
    'Baseline\n(application)': imp_mean[[f for f in imp_mean.index if f not in all_new_features]].sum(),
    'previous\napplication': imp_mean[[f for f in imp_mean.index if f in prev_features]].sum(),
    'installments\npayments': imp_mean[[f for f in imp_mean.index if f in ins_features]].sum(),
    'POS_CASH\nbalance': imp_mean[[f for f in imp_mean.index if f in pos_features]].sum(),
}
group_colors = ['#95a5a6', '#e74c3c', '#3498db', '#2ecc71']
bars = axes[1].bar(group_imp.keys(), group_imp.values(), color=group_colors)
axes[1].set_ylabel('Total Feature Importance')
axes[1].set_title('Importance by Source Table')
for bar, val in zip(bars, group_imp.values()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{val:.0f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
os.makedirs('../analysis', exist_ok=True)
plt.savefig('../analysis/enhanced_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# Detailed stats
print(f"\n{'='*60}")
print(f"FEATURE IMPORTANCE THEO NGUỒN")
print(f"{'='*60}")
total_imp = imp_mean.sum()
for group_name, feature_list_grp in [
    ('Baseline (application)', [f for f in imp_mean.index if f not in all_new_features]),
    ('previous_application', [f for f in imp_mean.index if f in prev_features]),
    ('installments_payments', [f for f in imp_mean.index if f in ins_features]),
    ('POS_CASH_balance', [f for f in imp_mean.index if f in pos_features]),
]:
    grp_imp = imp_mean[feature_list_grp]
    pct = grp_imp.sum() / total_imp * 100
    in_top30 = sum(1 for f in feature_list_grp if f in top30.index)
    print(f"\n  {group_name}:")
    print(f"    Features: {len(feature_list_grp)}, Importance: {pct:.1f}%")
    print(f"    In Top 30: {in_top30}")
    print(f"    Top 5: {', '.join(grp_imp.head(5).index.tolist())}")

## 10. SHAP Explainability

In [ ]:
import shap

sample_size = min(2000, len(X))
X_sample = X.sample(sample_size, random_state=42)

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_sample)

if isinstance(shap_values, list):
    shap_vals = shap_values[1]
else:
    shap_vals = shap_values

print(f'SHAP values computed for {sample_size} samples')

In [ ]:
# SHAP Summary Plot
plt.figure(figsize=(12, 10))
shap.summary_plot(shap_vals, X_sample, max_display=25, show=False)
plt.title('SHAP Summary — Enhanced Model (114 features)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Bar Plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_vals, X_sample, plot_type='bar', max_display=25, show=False)
plt.title('Mean |SHAP Value| — Enhanced Model', fontsize=14)
plt.tight_layout()
plt.show()

## 11. FICO Scoring & Probability Analysis

In [ ]:
# ============================================================
# PHÂN PHỐI XÁC SUẤT & FICO SCORING
# ============================================================
y_arr = y.values
eps = 1e-7

# FICO parameters
PDO = 40
BASE_SCORE = 600
FACTOR = PDO / np.log(2)

fico_scores = BASE_SCORE - FACTOR * np.log(
    np.clip(oof_preds, eps, 1-eps) / (1 - np.clip(oof_preds, eps, 1-eps))
)
fico_scores = np.clip(fico_scores, 300, 850)

# Distribution stats
print("=" * 60)
print("PHÂN PHỐI FICO SCORE (Enhanced Model)")
print("=" * 60)
fico_series = pd.Series(fico_scores)
desc = fico_series.describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
for idx, val in desc.items():
    print(f"  {idx:<12} {val:>8.1f}")

print(f"\n  By actual target:")
for label, name in [(0, "Good"), (1, "Bad")]:
    mask = y_arr == label
    sub = fico_scores[mask]
    print(f"    {name}: Mean={sub.mean():.1f}  Median={np.median(sub):.1f}  Std={sub.std():.1f}")

# Tier distribution
print(f"\n{'='*60}")
print(f"TIER DISTRIBUTION")
print(f"{'='*60}")
tiers = [
    ("Exceptional (800-850)", 800, 850),
    ("Very Good (740-799)", 740, 799),
    ("Good (670-739)", 670, 739),
    ("Fair (580-669)", 580, 669),
    ("Poor (300-579)", 300, 579),
]
print(f"\n{'Tier':<25} {'Count':>8} {'%':>8} {'Default%':>10}")
print("-" * 55)
for name, lo, hi in tiers:
    mask = (fico_scores >= lo) & (fico_scores <= hi)
    n = mask.sum()
    pct = 100 * n / len(fico_scores)
    def_rate = 100 * y_arr[mask].mean() if n > 0 else 0
    print(f"  {name:<23} {n:>8,} {pct:>7.1f}% {def_rate:>9.2f}%")

In [ ]:
# ---- Visualization ----
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Probability distribution
axes[0, 0].hist(oof_preds[y_arr==0], bins=100, alpha=0.7, label='Good (0)', color='green', density=True)
axes[0, 0].hist(oof_preds[y_arr==1], bins=100, alpha=0.7, label='Bad (1)', color='red', density=True)
axes[0, 0].set_title('Distribution of P(default)')
axes[0, 0].set_xlabel('Predicted Probability')
axes[0, 0].legend()

# FICO distribution
axes[0, 1].hist(fico_scores[y_arr==0], bins=80, alpha=0.7, label='Good (0)', color='green', density=True)
axes[0, 1].hist(fico_scores[y_arr==1], bins=80, alpha=0.7, label='Bad (1)', color='red', density=True)
axes[0, 1].set_title('FICO Score Distribution')
axes[0, 1].set_xlabel('FICO Score (300-850)')
axes[0, 1].axvline(x=580, color='orange', linestyle='--', label='Fair/Poor')
axes[0, 1].axvline(x=670, color='blue', linestyle='--', label='Good')
axes[0, 1].axvline(x=740, color='purple', linestyle='--', label='Very Good')
axes[0, 1].legend(fontsize=8)

# ROC Curve
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y, oof_preds)
axes[1, 0].plot(fpr, tpr, color='#e74c3c', linewidth=2, label=f'AUC={overall_auc:.4f}')
axes[1, 0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1, 0].set_title('ROC Curve')
axes[1, 0].set_xlabel('FPR')
axes[1, 0].set_ylabel('TPR')
axes[1, 0].legend()

# Boxplot by target
data_box = [fico_scores[y_arr==0], fico_scores[y_arr==1]]
bp = axes[1, 1].boxplot(data_box, labels=['Good (0)', 'Bad (1)'], patch_artist=True)
bp['boxes'][0].set_facecolor('lightgreen')
bp['boxes'][1].set_facecolor('lightcoral')
axes[1, 1].set_title('FICO Score by Target')
axes[1, 1].set_ylabel('FICO Score')

plt.tight_layout()
plt.savefig('../analysis/enhanced_model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Probability Calibration

In [ ]:
# ============================================================
# ISOTONIC CALIBRATION
# ============================================================
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve

iso_reg = IsotonicRegression(out_of_bounds='clip')
iso_reg.fit(oof_preds, y)
calibrated_preds = iso_reg.predict(oof_preds)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before
frac_pos_before, mean_pred_before = calibration_curve(y, oof_preds, n_bins=15)
axes[0].plot(mean_pred_before, frac_pos_before, 's-', color='#e74c3c', linewidth=2, label='Before')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_title('Before Calibration')
axes[0].set_xlabel('Mean Predicted Probability')
axes[0].set_ylabel('Fraction of Positives')
axes[0].legend()

# After
frac_pos_after, mean_pred_after = calibration_curve(y, calibrated_preds, n_bins=15)
axes[1].plot(mean_pred_after, frac_pos_after, 's-', color='#2ecc71', linewidth=2, label='After (Isotonic)')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1].set_title('After Isotonic Calibration')
axes[1].set_xlabel('Mean Predicted Probability')
axes[1].set_ylabel('Fraction of Positives')
axes[1].legend()

plt.tight_layout()
plt.show()

auc_after = roc_auc_score(y, calibrated_preds)
print(f'AUC before: {overall_auc:.6f}')
print(f'AUC after:  {auc_after:.6f}')
print('→ Calibration preserves ranking, chỉ cải thiện probability accuracy')

## 13. Save Model Artifacts

In [ ]:
# ============================================================
# SAVE ALL ARTIFACTS → enhanced_model_artifacts/
# ============================================================
SAVE_DIR = Path('enhanced_model_artifacts')
SAVE_DIR.mkdir(exist_ok=True)

# 1. Model
joblib.dump(best_model, SAVE_DIR / 'lgbm_credit_scoring_enhanced.pkl')
print(f'✅ Model saved ({(SAVE_DIR / "lgbm_credit_scoring_enhanced.pkl").stat().st_size / 1024:.0f} KB)')

# 2. Feature names
with open(SAVE_DIR / 'feature_names.json', 'w') as f:
    json.dump(FINAL_FEATURES, f, indent=2)
print(f'✅ Feature names saved ({len(FINAL_FEATURES)} features)')

# 3. Label encoders
joblib.dump(label_encoders, SAVE_DIR / 'label_encoders.pkl')
print(f'✅ Label encoders saved ({len(label_encoders)} encoders)')

# 4. SHAP explainer
joblib.dump(explainer, SAVE_DIR / 'shap_explainer.pkl')
print(f'✅ SHAP explainer saved')

# 5. Isotonic calibrator
joblib.dump(iso_reg, SAVE_DIR / 'isotonic_calibrator.pkl')
print(f'✅ Isotonic calibrator saved')

# 6. Feature sets (for reference)
feature_sets = {
    'all_features': FINAL_FEATURES,
    'baseline_features': [f for f in FINAL_FEATURES if f not in all_new_features],
    'new_features': all_new_features,
    'prev_features': prev_features,
    'ins_features': ins_features,
    'pos_features': pos_features,
    'engineered': engineered,
}
with open(SAVE_DIR / 'feature_sets.json', 'w') as f:
    json.dump(feature_sets, f, indent=2)
print(f'✅ Feature sets saved')

print(f'\n📁 All artifacts saved to: {SAVE_DIR.absolute()}')

## 14. Export Dataset

In [ ]:
# ============================================================
# EXPORT DATASET TỔNG HỢP → CSV
# ============================================================
export_dir = os.path.abspath('../output')
os.makedirs(export_dir, exist_ok=True)

export_path = os.path.join(export_dir, 'credit_scoring_enhanced_dataset.csv')
data.to_csv(export_path, index=False)
print(f"✅ Dataset exported: {export_path}")
print(f"   Shape: {data.shape}")
print(f"   Features: {len(FINAL_FEATURES)} (48 baseline + {len(all_new_features)} supplementary)")
print(f"   File size: {os.path.getsize(export_path) / 1024 / 1024:.1f} MB")

## 15. Model Summary

### So sánh 2 phiên bản

| Metric | Notebook gốc (no history) | Notebook Enhanced |
|--------|---------------------------|-------------------|
| **Data sources** | 1 bảng (application) | 4 bảng (app + prev + ins + pos) |
| **Features** | 48 | 114 |
| **OOF AUC** | 0.767 | **0.781** |
| **Improvement** | — | **+1.4% AUC** |
| **SHAP** | EXT_SOURCE dominant | EXT_SOURCE + supplementary data |

### Khi nào dùng notebook nào?

| Trường hợp | Notebook |
|------------|----------|
| Khách hàng **mới hoàn toàn** (chưa từng vay) | `credit_scoring_no_history.ipynb` — 48 features |
| Khách hàng **có lịch sử** vay/trả trước đó | `credit_scoring_enhanced.ipynb` — 114 features |
| Demo webapp nhanh, nhẹ | Notebook gốc |
| Accuracy cao nhất có thể | Notebook Enhanced |

In [ ]:
# ============================================================
# MODEL SUMMARY
# ============================================================
model_size = os.path.getsize(SAVE_DIR / 'lgbm_credit_scoring_enhanced.pkl') / 1024

print('╔══════════════════════════════════════════════════════════════╗')
print('║     ENHANCED CREDIT SCORING MODEL — SUMMARY                ║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║  Model:           LightGBM Classifier                      ║')
print(f'║  Features:        {len(FINAL_FEATURES):3d} (48 baseline + 66 supplementary)    ║')
print(f'║  Data sources:    application + prev_app + instal + POS    ║')
print(f'║  Training data:   {len(X):,} samples                      ║')
print(f'║  OOF AUC:         {overall_auc:.6f}                              ║')
print(f'║  Mean Fold AUC:   {np.mean(fold_aucs):.6f} ± {np.std(fold_aucs):.6f}             ║')
print(f'║  Model size:      {model_size:.0f} KB                              ║')
print(f'║  vs Baseline:     +{(overall_auc - 0.7673)*100:.2f}% AUC improvement              ║')
print('╠══════════════════════════════════════════════════════════════╣')
print('║  Artifacts:       enhanced_model_artifacts/                 ║')
print('║  Dataset:         output/credit_scoring_enhanced_dataset.csv║')
print('╚══════════════════════════════════════════════════════════════╝')